# Data Cleaning — events.app_raw_table

Sanity checks and JSON-field parsing on the raw event table.

1. Volume per day (look for ingestion gaps).
2. Duplicate event_id detection.
3. JSON field exploration (event_metadata, user_metadata).

In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project='hopeful-list-429812-f3')

In [ ]:
df_volume = client.query('''
SELECT DATE(timestamp) AS d, COUNT(*) AS n
FROM `hopeful-list-429812-f3.events.app_raw_table`
WHERE DATE(timestamp) BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY) AND CURRENT_DATE()
GROUP BY d ORDER BY d
''').to_dataframe()
df_volume.tail()

In [ ]:
df_dups = client.query('''
SELECT event_id, COUNT(*) AS dup_count
FROM `hopeful-list-429812-f3.events.app_raw_table`
WHERE DATE(timestamp) >= DATE_SUB(CURRENT_DATE(), INTERVAL 7 DAY)
GROUP BY event_id HAVING COUNT(*) > 1
ORDER BY dup_count DESC LIMIT 20
''').to_dataframe()
df_dups

In [ ]:
df_steps = client.query('''
SELECT JSON_VALUE(event_metadata, '$.step_name') AS step_name, COUNT(*) AS n
FROM `hopeful-list-429812-f3.events.app_raw_table`
WHERE event_name = 'onboarding_step'
  AND DATE(timestamp) >= DATE_SUB(CURRENT_DATE(), INTERVAL 7 DAY)
GROUP BY step_name ORDER BY n DESC
''').to_dataframe()
df_steps